In [ ]:
# Package installation

%pip install "transformers>=4.51.0" torch torchvision accelerate \
            "qwen-vl-utils>=0.0.10" pillow pandas openpyxl matplotlib tqdm -q

In [ ]:
# Necessary imports

import sys
import os
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib     import Path
from datetime    import datetime
 
warnings.filterwarnings("ignore")
 
# Add parent directory to path so we can import the package
# (needed when running from the same folder as navigation_system/)
sys.path.insert(0, str(Path(__file__).parent if "__file__" in dir() else Path.cwd()))
 
from navigation_system               import NavigationSystem
from navigation_system.model_loader  import ModelWrapper
from navigation_system.goal_parser   import GoalParser
from navigation_system.memory        import NavigationMemory
from navigation_system.action_generator import ActionGenerator
 
print("✅ Imports done")

In [ ]:
# Configuration

MODEL_NAME = "Qwen/Qwen3-VL-8B-Instruct"

IMAGE_DIR    = Path("nav_images")
SEQUENCE_DIR = IMAGE_DIR / "sequence"


#check the file


In [ ]:
# Loading the model

model = ModelWrapper(MODEL_NAME).load()
print(f"\n✅ Model ready: {MODEL_NAME}")

In [ ]:
# Goal Parser

# Tests whether the parser correctly breaks down a room code
# into building / floor / room and creates a sensible 3-step plan.
 
def demo_goal_parser(room_codes: list[str]) -> pd.DataFrame:
    """Run the goal parser on several room codes and display results."""
    parser = GoalParser(model)
    rows   = []
 
    print(f"\n{'='*55}")
    print(f"  STAGE 2 DEMO — Goal Parser")
    print(f"{'='*55}\n")
 
    for code in room_codes:
        print(f"── Parsing: {code} ──")
        try:
            plan = parser.parse(code)
 
            rows.append({
                "room_code"  : code,
                "building"   : plan.building,
                "floor"      : plan.floor,
                "room"       : plan.room,
                "step1_goal" : plan.steps[0].goal,
                "step2_goal" : plan.steps[1].goal,
                "step3_goal" : plan.steps[2].goal,
                "parse_ok"   : True,
                "error"      : "",
            })
 
        except Exception as e:
            print(f"  ❌ Error: {e}")
            rows.append({
                "room_code"  : code,
                "building"   : "",
                "floor"      : "",
                "room"       : "",
                "step1_goal" : "",
                "step2_goal" : "",
                "step3_goal" : "",
                "parse_ok"   : False,
                "error"      : str(e),
            })
 
    df = pd.DataFrame(rows)
 
    print(f"\n{'─'*55}")
    print("  Parser results:")
    print(df[["room_code", "building", "floor", "room", "parse_ok"]].to_string(index=False))
    n_ok = df["parse_ok"].sum()
    print(f"\n  Parsed correctly: {n_ok}/{len(df)}")
    return df
 
 
parser_results = demo_goal_parser(COMPARISON_ROOMS)

In [ ]:
# Memory

# Pick any one image from your sequence folder and test
# whether the memory module extracts useful navigation info.
 
def demo_memory_single(image_path: str):
    """Run the memory extractor on one image and show what it finds."""
    from navigation_system.goal_parser import NavigationStep
 
    # Create a dummy step for testing
    dummy_step = NavigationStep(
        index    = 1,
        level    = "building",
        goal     = f"Find and enter Building C",
        look_for = "Building C signs, directory boards, building entrances",
        done_when= "Inside Building C",
        status   = "active",
    )
 
    memory = NavigationMemory(model)
    print(f"\n{'='*55}")
    print(f"  STAGE 1 DEMO — Memory Extractor")
    print(f"  Image: {Path(image_path).name}")
    print(f"  Goal : {dummy_step.goal}")
    print(f"{'='*55}\n")
 
    entry = memory.update(image_path, dummy_step)
 
    if entry:
        print(f"  ✅ Memory entry created:")
        print(f"     Content: {entry.content}")
    else:
        print(f"  — Nothing navigation-useful found in this image")
        print(f"     (This is correct if the image shows an empty corridor)")
 
    return entry
 
 
# ── Run on first available image in the sequence folder ────────────────────────
seq_images = sorted(
    f for f in SEQUENCE_DIR.iterdir()
    if f.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
)
 
if seq_images:
    demo_memory_single(str(seq_images[0]))
else:
    print("⚠️  No images in nav_images/sequence/ yet.")
    print("   Add images then re-run this cell.")
 

In [ ]:
# Action Generation

# Tests whether the action generator produces a sensible,
# validated action when given one image and the current plan state.
 
def demo_action_generator(image_path: str, room_code: str = TARGET_ROOM):
    """Run the action generator on one image and display the result."""
    # Build plan and memory
    parser  = GoalParser(model)
    plan    = parser.parse(room_code)
    memory  = NavigationMemory(model)
 
    # Add the image to memory first
    memory.update(image_path, plan.current_step)
 
    # Generate action
    gen    = ActionGenerator(model)
    action = gen.generate(image_path, memory, plan)
 
    print(f"\n{'='*55}")
    print(f"  STAGE 3 DEMO — Action Generator")
    print(f"  Image     : {Path(image_path).name}")
    print(f"  Room code : {room_code}")
    print(f"  Nav step  : Step {plan.current_step_number}/3  [{plan.current_step.level}]")
    print(f"{'='*55}")
    print(f"\n  {action.display()}")
    return action
 
 
if seq_images:
    # Use first image for demo
    demo_action_generator(str(seq_images[0]))
else:
    print("⚠️  Add images to nav_images/sequence/ first.")

In [ ]:
# Full navigation run

# Runs the complete system over your entire image sequence.
# This is the main experiment for your paper.
 
def run_full_navigation(
    room_code   : str        = TARGET_ROOM,
    images      : list[str]  = None,
    save_prefix : str        = "",
) -> object:
    """
    Run the complete navigation system over a sequence of images.
 
    Args:
        room_code   : Target room, e.g. "C2.005"
        images      : List of image paths. If None, uses all images
                      in SEQUENCE_DIR in sorted order.
        save_prefix : Prefix for output files.
 
    Returns:
        NavigationLog
    """
    if images is None:
        exts   = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
        images = sorted(
            str(f) for f in SEQUENCE_DIR.iterdir()
            if f.suffix.lower() in exts
        )
 
    if not images:
        print("⚠️  No images found. Add photos to nav_images/sequence/ first.")
        return None
 
    print(f"\nRunning navigation: {room_code}")
    print(f"Images to process: {len(images)}")
 
    nav    = NavigationSystem(model)
    prefix = f"{save_prefix}_{room_code}" if save_prefix else room_code
    log    = nav.navigate(
        room_code      = room_code,
        image_sequence = images,
        save_log       = True,
        log_path       = str(OUTPUT_DIR / f"nav_log_{prefix}.json"),
    )
 
    return log
 
 
# ── Run it ────────────────────────────────────────────────────────────────────
nav_log = run_full_navigation(TARGET_ROOM)


In [ ]:
# Visualization

def visualise_run(log, max_images: int = 12):
    """
    Show a grid of images with the action taken at each step.
    Also prints a memory timeline.
    """
    if log is None:
        print("No log to display.")
        return
 
    records = log.records[:max_images]
    n       = len(records)
    if n == 0:
        print("No records in log.")
        return
 
    cols = min(n, 4)
    rows = (n + cols - 1) // cols
 
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4.5, rows * 4))
    axes_flat = axes.flatten() if hasattr(axes, "flatten") else [axes]
 
    level_colors = {"building": "#3498db", "floor": "#e67e22", "room": "#27ae60"}
 
    for i, rec in enumerate(records):
        ax = axes_flat[i]
 
        try:
            from PIL import Image as PILImage
            img = PILImage.open(rec.image_path)
            ax.imshow(img)
        except Exception:
            ax.set_facecolor("#f0f0f0")
            ax.text(0.5, 0.5, "Image\nnot found",
                    ha="center", va="center", transform=ax.transAxes)
 
        ax.axis("off")
 
        # Step badge
        col = level_colors.get(
            log.plan.steps[rec.nav_step - 1].level, "#888"
        ) if rec.nav_step <= len(log.plan.steps) else "#888"
 
        ax.set_title(
            f"#{rec.image_num}  S{rec.nav_step}",
            fontsize=8, color="white",
            bbox=dict(boxstyle="round,pad=0.2", facecolor=col, alpha=0.85),
        )
 
        # Action label at bottom
        valid_sym = "✅" if rec.action.is_valid else "⚠️"
        done_sym  = " →DONE" if rec.step_advanced else ""
        label     = f"{valid_sym} {rec.action.raw}{done_sym}"
        ax.text(
            0.5, -0.03, label,
            transform  = ax.transAxes,
            fontsize   = 7,
            ha         = "center",
            va         = "top",
            color      = "#1a1a1a",
            wrap       = True,
        )
 
    # Hide unused axes
    for j in range(i + 1, len(axes_flat)):
        axes_flat[j].set_visible(False)
 
    status = "✅ SUCCESS" if log.success else "❌ INCOMPLETE"
    fig.suptitle(
        f"Navigation: {log.room_code}   {status}   "
        f"Steps reached: {log.final_step_reached}/3",
        fontsize    = 12,
        fontweight  = "bold",
        y           = 1.01,
    )
    plt.tight_layout()
    chart_path = OUTPUT_DIR / f"nav_chart_{log.room_code}.png"
    plt.savefig(str(chart_path), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"📊 Chart saved: {chart_path}")
 
    # ── Memory timeline ────────────────────────────────────────────────────────
    print(f"\n{'─'*55}")
    print(f"  Memory timeline ({len(log.memory_entries)} useful observations):")
    if log.memory_entries:
        for e in log.memory_entries:
            print(f"  [Img #{e.image_num} | Step {e.step_num}] {e.content}")
    else:
        print("  (none — no navigation cues were detected)")
 
 
if nav_log:
    visualise_run(nav_log)

In [ ]:
# Run the same image sequence for several different room codes.
# This is the core experiment for your paper:
# "Does the system behave differently depending on the target room?"
# (It should — the 3-step plan and decision logic change per room.)
 
def run_comparison(
    room_codes : list[str],
    images     : list[str] = None,
) -> pd.DataFrame:
    """
    Run navigation for multiple room codes and collect summary stats.
 
    Returns a DataFrame with one row per room code.
    """
    if images is None:
        exts   = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
        images = sorted(
            str(f) for f in SEQUENCE_DIR.iterdir()
            if f.suffix.lower() in exts
        )
 
    if not images:
        print("⚠️  Add images to nav_images/sequence/ first.")
        return pd.DataFrame()
 
    print(f"\n{'='*55}")
    print(f"  MULTI-ROOM COMPARISON")
    print(f"  Rooms  : {room_codes}")
    print(f"  Images : {len(images)} per run")
    print(f"{'='*55}\n")
 
    rows = []
    for room in room_codes:
        print(f"\n{'─'*40}")
        print(f"  Room: {room}")
        print(f"{'─'*40}")
 
        nav = NavigationSystem(model)
        log = nav.navigate(
            room_code      = room,
            image_sequence = images,
            save_log       = True,
            log_path       = str(OUTPUT_DIR / f"nav_log_{room}.json"),
        )
 
        # Counts
        total_actions   = len(log.records)
        valid_actions   = sum(1 for r in log.records if r.action.is_valid)
        high_conf       = sum(1 for r in log.records if r.action.confidence == "HIGH")
        mem_entries     = len(log.memory_entries)
 
        rows.append({
            "room_code"       : room,
            "success"         : log.success,
            "steps_reached"   : log.final_step_reached,
            "images_processed": log.total_images_processed,
            "total_actions"   : total_actions,
            "valid_actions"   : valid_actions,
            "invalid_actions" : total_actions - valid_actions,
            "high_confidence" : high_conf,
            "memory_entries"  : mem_entries,
        })
 
    df = pd.DataFrame(rows)
    print(f"\n{'='*55}")
    print("  COMPARISON SUMMARY:")
    print(df.to_string(index=False))
 
    # Save comparison table
    df.to_excel(str(OUTPUT_DIR / "comparison_results.xlsx"), index=False)
    print(f"\n  Saved: {OUTPUT_DIR}/comparison_results.xlsx")
 
    return df
 
 
# ── Uncomment to run the comparison ──────────────────────────────────────────
# comparison_df = run_comparison(COMPARISON_ROOMS)

In [ ]:
# Plot comparison chart

def plot_comparison(df: pd.DataFrame):
    """Bar chart comparing steps reached and valid action rate per room."""
    if df.empty:
        print("No data to plot.")
        return
 
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
 
    rooms = df["room_code"].tolist()
    x     = range(len(rooms))
    w     = 0.35
 
    # Left: steps reached
    bars1 = ax1.bar(x, df["steps_reached"], color="#3498db", alpha=0.85, label="Steps reached")
    ax1.axhline(y=3, color="#e74c3c", linestyle="--", linewidth=1.2, label="Max (3)")
    ax1.set_xticks(list(x))
    ax1.set_xticklabels(rooms, fontsize=9)
    ax1.set_ylabel("Navigation steps reached (out of 3)")
    ax1.set_ylim(0, 3.8)
    ax1.legend(fontsize=8)
    ax1.set_title("Steps Reached per Room", fontsize=11)
    ax1.grid(axis="y", alpha=0.3, linestyle="--")
    for bar in bars1:
        h = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width() / 2, h + 0.05,
                 f"{h:.0f}", ha="center", va="bottom", fontsize=9)
 
    # Right: valid vs invalid actions
    valid   = df["valid_actions"].tolist()
    invalid = df["invalid_actions"].tolist()
    ax2.bar([i - w/2 for i in x], valid,   w, label="Valid",   color="#27ae60", alpha=0.85)
    ax2.bar([i + w/2 for i in x], invalid, w, label="Invalid", color="#e74c3c", alpha=0.85)
    ax2.set_xticks(list(x))
    ax2.set_xticklabels(rooms, fontsize=9)
    ax2.set_ylabel("Number of actions")
    ax2.legend(fontsize=8)
    ax2.set_title("Valid vs Invalid Actions per Room", fontsize=11)
    ax2.grid(axis="y", alpha=0.3, linestyle="--")
 
    for ax in (ax1, ax2):
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
 
    plt.suptitle(
        f"Navigation System — Room Comparison\nModel: {MODEL_NAME}",
        fontsize=12, y=1.02,
    )
    plt.tight_layout()
    chart_path = OUTPUT_DIR / "comparison_chart.png"
    plt.savefig(str(chart_path), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"📊 Saved: {chart_path}")
 
 
# ── Uncomment after running Cell 11 ──────────────────────────────────────────
# plot_comparison(comparison_df)

In [ ]:
# Test each stage individually

# Run these to verify each stage is working before a full run.
 
def quick_check_stage2(room_code: str = TARGET_ROOM):
    """Quick test: can the parser handle this room code?"""
    parser = GoalParser(model)
    plan   = parser.parse(room_code)
    assert plan.building, "Building not parsed"
    assert plan.floor,    "Floor not parsed"
    assert plan.room,     "Room not parsed"
    assert len(plan.steps) == 3, "Should have exactly 3 steps"
    print(f"✅ Stage 2 OK — parsed {room_code}")
    return plan
 
 
def quick_check_stage1(image_path: str, room_code: str = TARGET_ROOM):
    """Quick test: does memory extractor work on this image?"""
    from navigation_system.goal_parser import NavigationStep
 
    step = NavigationStep(
        index    = 1,
        level    = "building",
        goal     = f"Find Building {room_code[0]}",
        look_for = "Building signs, directory boards",
        done_when= "Inside the correct building",
        status   = "active",
    )
    mem   = NavigationMemory(model)
    entry = mem.update(image_path, step)
    print(f"✅ Stage 1 OK — found: {entry.content[:60] if entry else 'nothing (OK)'}")
    return entry
 
 
def quick_check_stage3(image_path: str, room_code: str = TARGET_ROOM):
    """Quick test: does action generator produce a valid action?"""
    parser = GoalParser(model)
    plan   = parser.parse(room_code)
    mem    = NavigationMemory(model)
    gen    = ActionGenerator(model)
    action = gen.generate(image_path, mem, plan)
    assert action.name in ActionGenerator.VALID_ACTIONS, \
        f"Unknown action: {action.name}"
    print(f"✅ Stage 3 OK — action: {action.raw} [{action.confidence}]")
    return action
 
 
# ── Run quick checks if images are available ──────────────────────────────────
if seq_images:
    print("Running quick checks on first image...\n")
    quick_check_stage2(TARGET_ROOM)
    quick_check_stage1(str(seq_images[0]))
    quick_check_stage3(str(seq_images[0]))
    print("\n✅ All stages operational — ready for full run (Cell 9)")
else:
    print("⚠️  Add images to run quick checks.")